In [1]:
import pickle
from FlyOutput import FlyOutput
import Plotters
import plotly.graph_objects as go
import numpy as np  
import Utils
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import time
import pandas as pd
from PoseEstimation import PoseEstimation
import matplotlib.cm as cm
from scipy.signal import savgol_filter

%matplotlib qt





path_output = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'
model_name = 'fly_model_to_fly'
file_name = 'fly_model'
dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model.pkl'
image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/'
output_angles_weights_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/fly_model_to_fly/fly_model_results.pkl'
input_dir = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/fly_model_to_fly'



# path_output = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'
# model_name = 'cornell_mov9'
# file_name = 'fly_cornell_mov9'
# dict_path  = 'G://My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov9_cornell/dict/frames_model_cornell.pkl'
# image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov9_cornell/'
# output_angles_weights_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'+model_name+'/fly_model_results.pkl'
# input_dir = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/' + model_name + '/'




path_angles = f'{path_output}/{model_name}/{file_name}_angles.pkl'
path_results = f'{path_output}/{model_name}/{file_name}_angles.pkl'


# download model_run localy
frame0 = 1430
path = [f'{input_dir}/results/{frame0}/{file_name}_results.pkl',f'{input_dir}/{file_name}_results.pkl']
idx = [os.path.exists(f'{path}') for path in path]
existing_paths = np.array(path)[idx]
output_angles_weights = None
if len(existing_paths) > 0:
    with open(existing_paths[0], 'rb') as handle:
        output_angles_weights = pickle.load(handle)

iteration = 1200


weight_flag = False

with open(dict_path,'rb') as f:
    frames = pickle.load(f)





angle_name = ['phi','theta','psi','phi','psi','yaw','pitch']
letedict = {'num_of_bins' : 20,'perc_wing_for_le' : 1, 'wing_length_snip':0.27}



def smooth_axes(data,window_length = 11,polyorder = 3):
    Y_smooth = savgol_filter(np.vstack(data), window_length=window_length,
                            polyorder=polyorder, axis=0, mode="interp")
    smooth_axis = [smooth for smooth in Y_smooth]
    return smooth_axis



with open(dict_path,'rb') as f:
    frames = pickle.load(f)
iterations = 1200

frames_fly = []
for frame in range(frame0,1830):
    frame_output = FlyOutput(image_path,frame,input_dir,output_angles_weights,frame0,iteration,file_name,letedict = letedict,deg = 0,skip_frames = 1,frames_dict = frames)
    frame_output.wings_parameters( frame_output.right_wing)
    frame_output.wings_parameters( frame_output.left_wing)

    frame_output.calculate_ybody(perc_wing_for_root = 0.3)
    frame_output.calculate_zbody()
    frame_output.calculate_chord(frame_output.right_wing)
    frame_output.calculate_chord(frame_output.left_wing)

    # frame_output.calculate_wing_angles(frame_output.right_wing, left = 0)
    # frame_output.calculate_wing_angles(frame_output.left_wing, left = 1)
    frames_fly.append(frame_output)

# xcont = [frame.xbody for frame in frames_fly]
# ycont = [frame.ybody for frame in frames_fly]
# zcont = [frame.zbody for frame in frames_fly]

# xsmooth = smooth_axes(xcont,window_length = 11,polyorder = 3)
# ysmooth = smooth_axes(ycont,window_length = 11,polyorder = 3)
# zsmooth = smooth_axes(zcont,window_length = 11,polyorder = 3)

# for frame,xsmooth,ysmooth,zsmooth in zip(frames_fly,xsmooth,ysmooth,zsmooth):
#     frame.xbody = xsmooth
#     frame.ybody = xsmooth
#     frame.zbody = xsmooth

# for frame_output in frames_fly:
#     frame_output.calculate_wing_angles(frame_output.right_wing)
#     frame_output.calculate_wing_angles(frame_output.left_wing)



In [7]:
0.15*5000/60


12.5

In [2]:
frame_output.right_wing['timings']

{'get_span': 0.001281500095501542,
 'get_le_te': 0.00014030002057552338,
 'get_le_te_bins_le': 0.01202180003747344,
 'get_le_te_bins_te': 0.01171789993532002,
 'approx_le': 0.005154700018465519,
 'check_direction_span_ransac': 1.6200006939470768e-05}

In [3]:


def project_on_plane( normal, vector):
    projected_vector = vector - np.atleast_2d(np.sum(normal*vector,axis = 1)).T*normal
    return projected_vector / np.atleast_2d(np.linalg.norm(projected_vector, axis = 1)).T


# phi


def calculate_body_vectors(frames_fly):

    sp_normal = np.vstack([frame.sp_normal for frame in frames_fly])
    xbody_mat = np.vstack([frame.xbody for frame in frames_fly])
    xbody_on_sp = project_on_plane(sp_normal,xbody_mat)
    ybody_on_sp = np.cross(xbody_on_sp,sp_normal)
    ybody_on_sp = ybody_on_sp / np.atleast_2d(np.linalg.norm(ybody_on_sp, axis = 1)).T
    return xbody_on_sp,ybody_on_sp,sp_normal


def calculate_phi(frames_fly, left,xbody_on_sp,ybody_on_sp,sp_normal):
    if left == 1:
        sign_left = -1
        le_ransac_mat = [frame.left_wing['le_ransac'][1] for frame in frames_fly]
        chord_mat = np.vstack([frame.left_wing['chord'] for frame in frames_fly])
        le_sp_normal = np.cross(sp_normal, le_ransac_mat)

    else:
        sign_left = 1
        le_ransac_mat = [frame.right_wing['le_ransac'][1] for frame in frames_fly]
        chord_mat = np.vstack([frame.right_wing['chord'] for frame in frames_fly])
        le_sp_normal = np.cross(le_ransac_mat,sp_normal)


    le_on_sp = project_on_plane(sp_normal,le_ransac_mat)
    xle = np.sum(le_on_sp*xbody_on_sp,axis = 1)
    yle = np.sum(le_on_sp*ybody_on_sp,axis = 1)
    phi = np.arctan2(sign_left*yle,xle) *180/np.pi
    theta = 90 - np.arccos(np.sum( sp_normal*le_ransac_mat, axis = 1))*180/np.pi

    sp_chord = np.cross(le_ransac_mat,le_sp_normal)
    sp_chord = sp_chord / np.atleast_2d(np.linalg.norm(sp_chord, axis = 1)).T



    ypsi = np.sum(chord_mat*sp_chord, axis = 1)
    xpsi = np.sum(chord_mat*le_sp_normal, axis = 1)
    psi = np.arctan2(sign_left*ypsi,xpsi) 
    psi = np.unwrap(psi)*180/np.pi+180
    return phi,theta,psi



left = 1
xbody_on_sp,ybody_on_sp,sp_normal = calculate_body_vectors(frames_fly)
phi = calculate_phi(frames_fly, left,xbody_on_sp,ybody_on_sp,sp_normal)

fig,ax = plt.subplots(3,1)
ax[0].plot(phi[0],'*')
ax[1].plot(phi[1],'*')
ax[2].plot(phi[2],'*')

left = 0
phi = calculate_phi(frames_fly, left,xbody_on_sp,ybody_on_sp,sp_normal)
ax[0].plot(phi[0],'*')
ax[1].plot(phi[1],'*')
ax[2].plot(phi[2],'*')


In [84]:
from scipy.spatial.transform import Rotation as Rscipy

def calculate_roll(yaw,pitch,roll,ybody):


    # compute yaw–pitch rotated intermediate axes (ey, ez)
    cy, sy = np.cos(yaw), np.sin(yaw)
    cp, sp = np.cos(pitch), np.sin(pitch)

    ey = np.array([-sy,  cy,   0 ])        # intermediate Y axis
    ez = np.array([ sp*cy, sp*sy, cp])     # intermediate Z axis

    # project body Y onto these axes
    Yy = np.dot(ybody, ey)
    Yz = np.dot(ybody, ez)

    return  np.arctan2(Yz, Yy)




In [90]:
body_cm = np.vstack([frame.body_cm for frame in frames_fly])
fig, ax = plt.subplots(3,1)
ax[0].plot(body_cm[:,0])
ax[1].plot(body_cm[:,1])
ax[2].plot(body_cm[:,2])

In [ ]:
xbody_mat = np.vstack([frame.xbody for frame in frames_fly])
ybody_mat = np.vstack([frame.ybody for frame in frames_fly])
zbody_mat = np.vstack([frame.zbody for frame in frames_fly])




In [86]:
xbody_mat = np.vstack([frame.xbody for frame in frames_fly])
z_lab = xbody_mat*0 + [0,0,1]

pitch = 90-np.arccos(np.sum(xbody_mat*z_lab,axis = 1))*180/np.pi
yaw = np.arctan2(xbody_mat[:,1],xbody_mat[:,0])*180/np.pi
roll = np.unwrap([calculate_roll(yaw,pitch,0,frame.ybody) for yaw,pitch,frame in zip(yaw*np.pi/180,pitch*np.pi/180,frames_fly)])


fig, ax = plt.subplots(3,1)
ax[0].plot(yaw)
ax[1].plot(pitch)
ax[2].plot(roll)

In [ ]:
for frame_output in frames_fly:
    frame_output.calculate_wing_angles(frame_output.right_wing)
    frame_output.calculate_wing_angles(frame_output.left_wing)



In [ ]:
left = 0

signy = -1 if left == 1 else 1
le_sp_normal = np.cross(frame_output.zbody, frame_output.right_wing['le_ransac'][1])
le_sp_normal = le_sp_normal / np.linalg.norm(le_sp_normal)


sp_chord = np.cross(frame_output.right_wing['le_ransac'][1],le_sp_normal)
sp_chord = sp_chord / np.linalg.norm(sp_chord)

ypsi = signy*np.dot(chord,sp_chord)
xpsi = np.dot(chord,le_sp_normal)

psi = np.arctan2(ypsi,xpsi) % 2*np.pi * 180/np.pi


2.7884488794995206

In [ ]:


v1 = frame_output.right_wing['tip_le'] - frame_output.right_wing['tip_mean']
v2 = frame_output.right_wing['root_le'] - frame_output.right_wing['tip_mean']
norm_le = np.cross(v1/np.linalg.norm(v1),v2/np.linalg.norm(v2))
chord = np.cross(frame_output.right_wing['le_ransac'][1],norm_le)
chord = chord /  np.linalg.norm(chord)


array([-0.4911162 , -0.03211599, -0.87050183])

In [12]:
frame_output.right_wing['tip_mean']

array([ 0.00999976, -0.01206176, -0.01017039])

In [8]:
frame_output.right_wing['root_le']

array([ 0.0115503 , -0.01092959, -0.01036546])

In [4]:
xcont = [frame.xbody for frame in frames_fly]
plt.plot(np.vstack(xcont),label='right wing le ransac y body')

plt.figure()
root_rwing = [frame.right_wing['root_le'] for frame in frames_fly]
plt.plot(np.vstack(root_rwing),label='right wing le ransac y body')

root_rwing = [frame.left_wing['root_le'] for frame in frames_fly]
plt.plot(np.vstack(root_rwing),label='right wing le ransac y body')

In [5]:
plt.figure()
phi = [frame.right_wing['psi'] for frame in frames_fly]
plt.plot(np.vstack(phi),'*',label='right wing phi')


In [6]:
plt.figure()
phi = [frame.right_wing['phi'] for frame in frames_fly]
plt.plot(np.vstack(phi),'*',label='right wing phi')


In [7]:
plt.figure()
phi = [frame.right_wing['theta'] for frame in frames_fly]
plt.plot(np.vstack(phi),'*',label='right wing phi')



In [2]:
plt.figure()
phi = [frame.right_wing['phi'] for frame in frames_fly]
plt.plot(np.vstack(phi),'*',label='right wing phi')



In [ ]:

np.dot((frames_fly[0].left_wing['le_ransac'][1] - frames_fly[0].left_wing['root_le']))


array([ 0.01385027, -0.00518564, -0.00540344])

In [ ]:
le_ransac,le_ransac2 = [],[]
ycont = []
ycont = [frame.ybody for frame in frames_fly]

    # le_ransac2.append(-frame.left_wing['le_ransac'][1])
from scipy.signal import savgol_filter
window_length = 11
polyorder = 3

Y_smooth = savgol_filter(np.vstack(ycont), window_length=window_length,
                         polyorder=polyorder, axis=0, mode="interp")

for frame,y_smooth in zip(frames_fly,Y_smooth):
    frame.ybody_smooth = y_smooth


In [20]:
plt.figure()
plt.plot(np.vstack(Y_smooth),label='right wing le ransac y body')
plt.plot(np.vstack(ycont),label='right wing le ransac y body')

In [4]:
frame_num = 0
body = frames_fly[frame_num].body
rw = frames_fly[frame_num].right_wing
lw = frames_fly[frame_num].left_wing

fig = go.Figure()
Plotters.scatter3d(fig,body,'green',4,'body',show_colorbar = False, opa=1)
# Plotters.scatter3d(fig,frame_output.right_wing,'red',4,'body',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['xyz_lab'],'cyan',4,'left wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['le_bins'],'blue',4,'le',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['te_bins'],'gray',4,'te',show_colorbar = False, opa=1)

Plotters.scatter3d(fig,rw['xyz_lab'],'pink',4,'right wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,rw['le_bins'],'red',4,'le_rw',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,rw['te_bins'],'magenta',4,'te_rw',show_colorbar = False, opa=1)


# Plotters.scatter3d(fig,np.vstack((root_wing_le_r,root_wing_le_l)),'black',10,'te_rw',show_colorbar = False, opa=1)



Plotters.scatter3d(fig,np.vstack((frames_fly[frame_num].body_cm,frames_fly[frame_num].body_cm + frames_fly[frame_num].ybody/600)),'green',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack((frames_fly[frame_num].body_cm,frames_fly[frame_num].body_cm + frames_fly[frame_num].xbody/600)),'red',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack((frames_fly[frame_num].body_cm,frames_fly[frame_num].body_cm + frames_fly[frame_num].zbody/600)),'blue',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')

fig.data[-1].line.width = 15
fig.data[-2].line.width = 15

fig.show()

In [26]:

frame_num  =70
rw = frames_fly[frame_num].right_wing
lw = frames_fly[frame_num].left_wing 
body = frames_fly[frame_num].body 

fig = go.Figure()
Plotters.scatter3d(fig,body - frames_fly[frame_num].body_cm,'green',4,'body',show_colorbar = False, opa=1)
# Plotters.scatter3d(fig,frame_output.right_wing,'red',4,'body',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['xyz_lab'] - frames_fly[frame_num].body_cm,'cyan',4,'left wing',show_colorbar = False, opa=1)

Plotters.scatter3d(fig,rw['xyz_lab'] - frames_fly[frame_num].body_cm,'pink',4,'right wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,tips,'black',4,'right wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,np.vstack(([0,0,0],[0,0,0] + axes[0,:]/1000)),'black',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack(([0,0,0],[0,0,0] + axes[1,:]/1000)),'black',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack(([0,0,0],[0,0,0] + axes[2,:]/1000)),'black',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack(([0,0,0],[0,0,0] + projected_le/1000)),'red',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack(([0,0,0] + rw['le_ransac'][0]- frames_fly[frame_num].body_cm,[0,0,0] + rw['le_ransac'][0]+ rw['le_ransac'][1]/1000- frames_fly[frame_num].body_cm)),'red',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')

fig.data[-1].line.width = 15
fig.data[-2].line.width = 15
fig.data[-3].line.width = 15

fig.show()

In [5]:
fig,ax = plt.subplots(1,3)
ax[0].plot(np.vstack(le_ransac)[:,0],'*')
# ax[0].plot(np.vstack(le_ransac2)[:,0],'*')


ax[1].plot(np.vstack(le_ransac)[:,1],'*')
# ax[1].plot(np.vstack(le_ransac2)[:,1],'*')


ax[2].plot(np.vstack(le_ransac)[:,2],'*')
# ax[2].plot(np.vstack(le_ransac2)[:,2],'*')


IndexError: index 1 is out of bounds for axis 1 with size 1

In [26]:
fig,ax = plt.subplots(1,3)
ax[0].plot(np.vstack(le_ransacr)[:,0],'*')
ax[0].plot(np.vstack(le_ransac2r)[:,0],'*')
ax[0].plot(np.vstack(le_ransac)[:,0],'*')
ax[0].plot(np.vstack(le_ransac2)[:,0],'*')



ax[1].plot(np.vstack(le_ransacr)[:,1],'*')
ax[1].plot(np.vstack(le_ransac2r)[:,1],'*')
ax[1].plot(np.vstack(le_ransac)[:,1],'*')
ax[1].plot(np.vstack(le_ransac2)[:,1],'*')


ax[2].plot(np.vstack(le_ransacr)[:,2],'*')
ax[2].plot(np.vstack(le_ransac2r)[:,2],'*')
ax[2].plot(np.vstack(le_ransac)[:,2],'*')
ax[2].plot(np.vstack(le_ransac2)[:,2],'*')


In [21]:
fig,ax = plt.subplots(1,3)

ax[0].plot(np.vstack(le_ransac)[:,0],'*')
ax[1].plot(np.vstack(le_ransac2)[:,0],'*')

In [27]:
frame_num = 0
rw = frames_fly[frame_num].right_wing
lw = frames_fly[frame_num].left_wing

fig = go.Figure()
Plotters.scatter3d(fig,frame_output.body,'green',4,'body',show_colorbar = False, opa=1)
# Plotters.scatter3d(fig,frame_output.right_wing,'red',4,'body',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['xyz_lab'],'cyan',4,'left wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['le_bins'],'blue',4,'le',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['te_bins'],'gray',4,'te',show_colorbar = False, opa=1)

Plotters.scatter3d(fig,rw['xyz_lab'],'pink',4,'right wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,rw['le_bins'],'red',4,'le_rw',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,rw['te_bins'],'magenta',4,'te_rw',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,np.vstack((rw['le_ransac'][0] - rw['le_ransac'][1]/1000,rw['le_ransac'][0] + rw['le_ransac'][1]/1000)),'black',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack((lw['le_ransac'][0] - lw['le_ransac'][1]/1000,lw['le_ransac'][0] + lw['le_ransac'][1]/1000)),'black',20,'le ransac',show_colorbar = False, opa=1, mode = 'lines')

fig.data[-1].line.width = 15
fig.data[-2].line.width = 15

fig.show()

In [2]:
import plotly.graph_objects as go
import numpy as np

# How many frames you want in the animation
n_frames = len(frames_fly)   # or set a smaller number if you want
def make_frame_fig(i):
    """Build a figure for a single frame index i and return its traces."""
    rw = frames_fly[i].right_wing
    lw = frames_fly[i].left_wing
    frame_output = frames_fly[i]      # <-- adjust to your actual structure

    tmp_fig = go.Figure()
    Plotters.scatter3d(tmp_fig, frame_output.body, 'green', 4, 'body',
                       show_colorbar=False, opa=1)

    Plotters.scatter3d(tmp_fig, lw['xyz_lab'], 'cyan', 4, 'left wing',
                       show_colorbar=False, opa=1)
    Plotters.scatter3d(tmp_fig, lw['le_bins'], 'blue', 4, 'le',
                       show_colorbar=False, opa=1)
    Plotters.scatter3d(tmp_fig, lw['te_bins'], 'gray', 4, 'te',
                       show_colorbar=False, opa=1)

    Plotters.scatter3d(tmp_fig, rw['xyz_lab'], 'pink', 4, 'right wing',
                       show_colorbar=False, opa=1)
    Plotters.scatter3d(tmp_fig, rw['le_bins'], 'red', 4, 'le_rw',
                       show_colorbar=False, opa=1)
    Plotters.scatter3d(tmp_fig, rw['te_bins'], 'magenta', 4, 'te_rw',
                       show_colorbar=False, opa=1)

    # LE RANSAC lines (right & left)
    Plotters.scatter3d(
        tmp_fig,
        np.vstack((rw['le_ransac'][0] - rw['le_ransac'][1]/1000,
                   rw['le_ransac'][0] + rw['le_ransac'][1]/1000)),
        'black', 20, 'le ransac rw',
        show_colorbar=False, opa=1, mode='lines'
    )
    Plotters.scatter3d(
        tmp_fig,
        np.vstack((lw['le_ransac'][0] - lw['le_ransac'][1]/1000,
                   lw['le_ransac'][0] + lw['le_ransac'][1]/1000)),
        'black', 20, 'le ransac',
        show_colorbar=False, opa=1, mode='lines'
    )
    
    Plotters.scatter3d(tmp_fig,np.vstack((frame_output.body_cm,frame_output.body_cm + frame_output.ybody/600)),'green',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
    Plotters.scatter3d(tmp_fig,np.vstack((frame_output.body_cm,frame_output.body_cm + frame_output.xbody/600)),'red',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
    Plotters.scatter3d(tmp_fig,np.vstack((frame_output.body_cm,frame_output.body_cm + frame_output.zbody/600)),'blue',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')


    # Make the last two traces thick lines (the RANSAC lines)
    tmp_fig.data[-1].line.width = 15
    tmp_fig.data[-2].line.width = 15
    tmp_fig.data[-3].line.width = 15
    tmp_fig.data[-4].line.width = 15
    tmp_fig.data[-5].line.width = 15

    return tmp_fig.data


# --- Build base figure (frame 0) ---
fig = go.Figure()
fig.add_traces(make_frame_fig(0))

# --- Build animation frames ---
frames = []
for i in range(n_frames):
    frame_traces = make_frame_fig(i)
    frames.append(go.Frame(data=frame_traces, name=str(i)))

fig.frames = frames

# --- Slider to control frames ---
sliders = [{
    "steps": [
        {
            "method": "animate",
            "label": str(i),
            "args": [
                [str(i)],
                {
                    "mode": "immediate",
                    "frame": {"duration": 0, "redraw": True},
                    "transition": {"duration": 0}
                }
            ],
        }
        for i in range(n_frames)
    ],
    "transition": {"duration": 0},
    "x": 0,
    "y": 0,
    "currentvalue": {"font": {"size": 16}, "prefix": "frame: ", "visible": True, "xanchor": "right"},
    "len": 1.0,
}]

# --- Play / Pause buttons ---
updatemenus = [{
    "type": "buttons",
    "showactive": False,
    "x": 0,
    "y": 1.1,
    "buttons": [
        {
            "label": "Play",
            "method": "animate",
            "args": [
                None,
                {
                    "frame": {"duration": 50, "redraw": True},
                    "fromcurrent": True,
                    "transition": {"duration": 0},
                },
            ],
        },
        {
            "label": "Pause",
            "method": "animate",
            "args": [
                [None],
                {
                    "frame": {"duration": 0, "redraw": False},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                },
            ],
        },
    ],
}]

fig.update_layout(
    updatemenus=updatemenus,
    sliders=sliders,
    scene=dict(
        aspectmode='data'
    )
)

fig.show()


In [20]:
import plotly.graph_objects as go
import numpy as np

n_frames = len(frames_fly)   # how many frames to animate

def build_traces(i):
    rw = frames_fly[i].right_wing
    lw = frames_fly[i].left_wing
    frame_output = frames_fly[i]   # adjust if needed

    fig = go.Figure()
    Plotters.scatter3d(fig, frame_output.body, 'green', 4, 'body', opa=1)

    Plotters.scatter3d(fig, lw['xyz_lab'], 'cyan', 4, 'lw', opa=1)
    Plotters.scatter3d(fig, lw['le_bins'], 'blue', 4, 'lw_le', opa=1)
    Plotters.scatter3d(fig, lw['te_bins'], 'gray', 4, 'lw_te', opa=1)

    Plotters.scatter3d(fig, rw['xyz_lab'], 'pink', 4, 'rw', opa=1)
    Plotters.scatter3d(fig, rw['le_bins'], 'red', 4, 'rw_le', opa=1)
    Plotters.scatter3d(fig, rw['te_bins'], 'magenta', 4, 'rw_te', opa=1)

    # RANSAC lines
    Plotters.scatter3d(
        fig,
        np.vstack((rw['le_ransac'][0] - rw['le_ransac'][1]/1000,
                   rw['le_ransac'][0] + rw['le_ransac'][1]/1000)),
        'black', 20, 'rw_ransac', opa=1, mode='lines'
    )
    Plotters.scatter3d(
        fig,
        np.vstack((lw['le_ransac'][0] - lw['le_ransac'][1]/1000,
                   lw['le_ransac'][0] + lw['le_ransac'][1]/1000)),
        'black', 20, 'lw_ransac', opa=1, mode='lines'
    )

    # thicker ransac lines
    fig.data[-1].line.width = 15
    fig.data[-2].line.width = 15

    return fig.data


# --- BASE FIGURE ---
fig = go.Figure()
fig.add_traces(build_traces(0))


# --- ANIMATION FRAMES ---
frames = []
for i in range(n_frames):
    frames.append(go.Frame(data=build_traces(i), name=str(i)))

fig.frames = frames


# --- SLIDER ---
fig.update_layout(
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "Frame: "},
        "pad": {"t": 30},
        "steps": [
            {
                "args": [[str(i)], {"frame": {"duration": 0, "redraw": True},
                                    "mode": "immediate"}],
                "label": str(i),
                "method": "animate"
            }
            for i in range(n_frames)
        ]
    }]
)


# --- PLAY / PAUSE BUTTONS ---
fig.update_layout(
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "x": 0.1,
        "y": 1.15,
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 50, "redraw": True},
                        "fromcurrent": True,
                        "transition": {"duration": 0}
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "transition": {"duration": 0}
                    }
                ]
            }
        ]
    }]
)

fig.show()
